# Open-Loop Rollout Diagnostic

For a Dreamer v1 world model, this notebook tests whether the model can predict
future observations using only the prior — i.e., without seeing observations
during the rollout. This is the diagnostic that matters for whether imagination-based
policy training will work, since the actor's training trajectory is a prior rollout.

**What we check:**
1. Closed-loop reconstruction is sharp (sanity).
2. Open-loop predictions track reality across `horizon` steps.
3. Open-loop error is bounded — doesn't explode.
4. Open-loop error beats a zero-dynamics baseline ("predict no change").
5. Per-dim error highlights which observation components the model handles well.
6. Reward prediction is accurate over the horizon (what the actor actually relies on).

In [ ]:
# Parameters — papermill injects overrides after this cell.
checkpoint_path = "checkpoints/default/latest.safetensors"
data_path       = "data/generated-pushes/"
data_format     = "rerun"   # "hdf5" or "rerun"
config_path     = None       # path to a YAML config override; default config used when None
num_episodes    = 20
burn_in         = 5
horizon         = 15
seed            = 0


## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np

from chuck_dreamer.config import load_config
from chuck_dreamer.dreamer import build_model
from chuck_dreamer.dreamer.mlx_model import feat
from chuck_dreamer.sim import PushingEnv
from chuck_dreamer.training.episode_loader import iter_episodes, processor_for

np.random.seed(seed)
mx.random.seed(seed)

print(f"checkpoint:    {checkpoint_path}")
print(f"data:          {data_path} ({data_format})")
print(f"num_episodes:  {num_episodes}")
print(f"burn_in:       {burn_in}")
print(f"horizon:       {horizon}")

## Load model and evaluation episodes

Checkpoints written by ``Trainer`` are flat ``.safetensors`` weight files; they don't
embed config or env shape. We rebuild the model the same way the trainer does
(``build_model`` with ``obs_shape``/``action_dim`` from a fresh ``PushingEnv``) and
then call ``model.load(...)``.

In [ ]:
config = load_config(config_path)

# Build env to derive obs/action shapes (matches Trainer.__init__).
env        = PushingEnv(config)
obs_shape  = env.observation_space.shape
action_dim = int(env.action_space.shape[0])

model = build_model(config, obs_shape=obs_shape, action_dim=action_dim)
# Switch to eval mode: load weights without rebuilding the optimizer.
model.training = False
model.load(checkpoint_path)

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"obs_shape={obs_shape}  action_dim={action_dim}  obs_mode={env.obs_mode}  act_mode={env.act_mode}")
print(f"RSSM dims: stoch={model.rssm.stoch_dim}, deter={model.rssm.deter_dim}")

In [ ]:
# Load evaluation episodes through the same processor the replay buffer uses.
processor = processor_for(env.obs_mode)
min_len   = burn_in + horizon

episodes: list[dict] = []
for raw in iter_episodes(data_path, format=data_format):
  ep = processor(raw)
  if ep["action"].shape[0] >= min_len:
    episodes.append(ep)
  if len(episodes) >= num_episodes:
    break

print(f"Using {len(episodes)} episodes (>= {min_len} steps)")

## Rollout helper

`open_loop_rollout` runs the world model in closed-loop for `burn_in` steps
(refining the latent with real observations) and then in open-loop for the
remaining steps (using only actions). We collect predicted observations and
rewards at every timestep.

The first ``burn_in`` actions are paired with the corresponding embeddings so the
posterior can refine the latent; afterwards we use the prior alone. Note the
RSSM convention: ``obs_step(prev_state, prev_action, embed_t)`` produces the
state at time ``t``, with ``embed_t`` being the embedding of ``obs[t]``.

In [ ]:
def open_loop_rollout(model, obs, actions, burn_in, deterministic=True):
  """Run a single-episode rollout. Returns predicted obs and rewards.

  obs:     (T+1, obs_dim) — obs[t] is the observation BEFORE action[t].
  actions: (T, action_dim)
  Returns dict of (T, ...) arrays for the post-action timesteps t=1..T.
  """
  T = actions.shape[0]
  obs_b = mx.array(obs[None])      # (1, T+1, obs_dim)
  act_b = mx.array(actions[None])  # (1, T, action_dim)

  embeds = model.encoder(obs_b)    # (1, T+1, embed_dim)
  state  = model.rssm.initial_state(1)
  states = []

  for t in range(T):
    if t < burn_in:
      # Posterior step: refine latent using embed of obs after action_t (i.e. obs[t+1]).
      state = model.rssm.obs_step(state, act_b[:, t], embeds[:, t + 1])
    else:
      if deterministic:
        # Prior rollout but without sampling noise: keep s = prior_mean.
        prev_s = state["s"]; prev_a = act_b[:, t]; prev_h = state["h"]
        x = model.rssm.pre_gru(mx.concatenate([prev_s, prev_a], axis=-1))
        h = model.rssm.gru(x[:, None, :], prev_h)[:, 0]
        prior_mean, prior_std = model.rssm._split_dist(model.rssm.prior_net(h))
        state = {"h": h, "s": prior_mean,
                 "prior_mean": prior_mean, "prior_std": prior_std}
      else:
        state = model.rssm.img_step(state, act_b[:, t])
    states.append(state)

  feats    = mx.stack([feat(s) for s in states], axis=1)  # (1, T, feat_dim)
  obs_pred = model.decoder(feats)
  rew_pred = model.reward_head(feats)
  mx.eval(obs_pred, rew_pred)

  return {
    "obs_pred": np.asarray(obs_pred[0]),
    "rew_pred": np.asarray(rew_pred[0]),
    "feats":    np.asarray(feats[0]),
  }

## Run rollouts

In [ ]:
results = []
for i, ep in enumerate(episodes):
  T = ep["action"].shape[0]
  out = open_loop_rollout(model, ep["obs"], ep["action"], burn_in)
  end = min(T, burn_in + horizon)
  results.append({
    "obs":      ep["obs"][1:end + 1],     # observation AFTER each action
    "obs_pred": out["obs_pred"][:end],
    "reward":   ep["reward"][:end],
    "rew_pred": out["rew_pred"][:end],
  })
print(f"Computed rollouts for {len(results)} episodes")

## Diagnostic 1 — Open-loop error vs horizon

Per-step L2 error between predicted and true observation, averaged over episodes.
The vertical line marks the burn-in boundary; before it the model sees observations,
after it the model is in open-loop mode.

We compare against a **zero-dynamics baseline**: "predict that the obs stays equal
to the last observed obs forever." For Lipschitz-continuous physical systems,
consecutive observations are close, so this is a decent baseline at short horizons
and a bad one at longer horizons. Your model should beat it after a few steps.

In [ ]:
def per_step_l2(pred, true):
  return np.linalg.norm(pred - true, axis=-1)

T_max = burn_in + horizon
errs_model = np.full((len(results), T_max), np.nan)
errs_zero  = np.full((len(results), T_max), np.nan)

for i, r in enumerate(results):
  T = r["obs"].shape[0]
  errs_model[i, :T] = per_step_l2(r["obs_pred"], r["obs"])
  if burn_in > 0:
    last_seen = r["obs"][burn_in - 1]
    zero_pred = np.tile(last_seen[None], (T, 1))
    errs_zero[i, :T] = per_step_l2(zero_pred, r["obs"])

mean_model = np.nanmean(errs_model, axis=0)
mean_zero  = np.nanmean(errs_zero,  axis=0)
std_model  = np.nanstd(errs_model,  axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(T_max)
ax.plot(xs, mean_model, label="world model", color="C0", linewidth=2)
ax.fill_between(xs, mean_model - std_model, mean_model + std_model,
                alpha=0.2, color="C0")
ax.plot(xs, mean_zero, label="zero-dynamics baseline", color="C3",
        linestyle="--", linewidth=1.5)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("step (action index)")
ax.set_ylabel("obs L2 error")
ax.set_title("Open-loop rollout error vs horizon")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Diagnostic 2 — Per-dimension open-loop error

For ``obs_mode="state"`` the obs vector is a fixed concatenation produced by
``StateVectorProcessor``: ``ee_pos (3) + ee_quat (4) + object_xy (2) + joint_qpos (n_joints)``.
Some dimensions are easy to predict (joint angles follow nearly directly from
the action sequence); others are hard (object position requires modeling contact
dynamics). Flat aggregate error can hide the dimensions that matter most for the
task.

In [ ]:
obs_dim = results[0]["obs"].shape[-1]
n_joints = obs_dim - 9   # ee_pos (3) + ee_quat (4) + object_xy (2) = 9 fixed dims

obs_dim_groups = {
  "ee_pos":     list(range(0, 3)),
  "ee_quat":    list(range(3, 7)),
  "object_xy":  list(range(7, 9)),
  "joint_qpos": list(range(9, obs_dim)),
}
obs_dim_labels = (
  ["ee_x", "ee_y", "ee_z"]
  + ["ee_qw", "ee_qx", "ee_qy", "ee_qz"]
  + ["obj_x", "obj_y"]
  + [f"q_{i}" for i in range(n_joints)]
)
assert len(obs_dim_labels) == obs_dim, (len(obs_dim_labels), obs_dim)

# Per-dim error in the open-loop region only.
per_dim = np.zeros(obs_dim)
counts  = np.zeros(obs_dim)
for r in results:
  T = r["obs"].shape[0]
  for t in range(burn_in, min(burn_in + horizon, T)):
    per_dim += np.abs(r["obs_pred"][t] - r["obs"][t])
    counts  += 1
per_dim /= np.maximum(counts, 1)

colors: list[str] = []
for i in range(obs_dim):
  for g_idx, (_, dims) in enumerate(obs_dim_groups.items()):
    if i in dims:
      colors.append(f"C{g_idx}")
      break
  else:
    colors.append("gray")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(obs_dim), per_dim, color=colors)
ax.set_xticks(np.arange(obs_dim))
ax.set_xticklabels(obs_dim_labels, rotation=45, ha="right")
ax.set_ylabel("mean |error| in open-loop region")
ax.set_title(f"Per-dimension prediction error, averaged over open-loop steps {burn_in}–{burn_in+horizon-1}")
for g_idx, name in enumerate(obs_dim_groups):
  ax.bar([], [], color=f"C{g_idx}", label=name)
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## Diagnostic 3 — Reward prediction error

The actor's training relies on the reward predictor's output during imagination.
If reward prediction degrades fast in open-loop, the actor's gradient signal is
wrong even if obs reconstruction looks fine. This is the metric most directly
predictive of whether the actor will train successfully.

In [ ]:
rew_errs = np.full((len(results), T_max), np.nan)
all_rewards = np.concatenate([r["reward"] for r in results])
mean_reward = float(all_rewards.mean()) if all_rewards.size else 0.0

err_array = np.full((len(results), T_max), np.nan)
for i, r in enumerate(results):
  T = r["reward"].shape[0]
  rew_errs[i, :T] = np.abs(r["rew_pred"] - r["reward"])
  err_array[i, :T] = np.abs(mean_reward - r["reward"])
mean_pred_err = np.nanmean(err_array, axis=0)

mean_rew = np.nanmean(rew_errs, axis=0)
std_rew  = np.nanstd(rew_errs,  axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(T_max)
ax.plot(xs, mean_rew, label="model reward error", color="C2", linewidth=2)
ax.fill_between(xs, mean_rew - std_rew, mean_rew + std_rew, alpha=0.2, color="C2")
ax.plot(xs, mean_pred_err, label="mean-predictor baseline",
        color="C3", linestyle="--", linewidth=1.5)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("step")
ax.set_ylabel("|predicted - true| reward")
ax.set_title("Reward prediction error vs horizon")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Diagnostic 4 — Single-episode trajectory comparison

Plot every dimension of obs across one episode, overlaying real vs predicted.
The eye is good at spotting when prediction visibly diverges. Use this as a
qualitative sanity check; the quantitative checks above are what you actually
track over training.

In [ ]:
ep_idx = 0   # change to inspect different episodes
r = results[ep_idx]
T, D = r["obs"].shape
ncols = 4
nrows = (D + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 2.5 * nrows),
                         sharex=True)
axes = np.atleast_2d(axes).flatten()
xs = np.arange(T)

for d in range(D):
  ax = axes[d]
  ax.plot(xs, r["obs"][:, d], label="real", color="C0", linewidth=1.5)
  ax.plot(xs, r["obs_pred"][:, d], label="pred", color="C1",
          linestyle="--", linewidth=1.5)
  ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5)
  ax.set_title(obs_dim_labels[d], fontsize=9)
  ax.grid(alpha=0.3)
for d in range(D, len(axes)):
  axes[d].axis("off")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle(f"Episode {ep_idx}: real vs predicted obs (dashed line = burn-in end)")
plt.tight_layout(); plt.show()

## Summary

The numbers below are the headline metrics. Track these across checkpoints to see
whether world model quality is improving over training.

In [ ]:
def _at(arr, idx):
  return float(arr[idx]) if 0 <= idx < arr.shape[0] else None

summary = {
  "checkpoint":                str(checkpoint_path),
  "num_episodes":              len(results),
  "burn_in":                   burn_in,
  "horizon":                   horizon,
  "obs_err_at_horizon_1":      _at(mean_model, burn_in),
  "obs_err_at_horizon_5":      _at(mean_model, burn_in + 4),
  "obs_err_at_horizon_15":     _at(mean_model, burn_in + 14),
  "zero_dyn_err_at_horizon_1": _at(mean_zero,  burn_in),
  "zero_dyn_err_at_horizon_5": _at(mean_zero,  burn_in + 4),
  "rew_err_at_horizon_5":      _at(mean_rew,    burn_in + 4),
  "rew_err_at_horizon_15":     _at(mean_rew,    burn_in + 14),
  "rew_baseline_at_horizon_5": _at(mean_pred_err, burn_in + 4),
}
print(json.dumps(summary, indent=2))